# 1. My Lane as an ML Task (Type)

-----

For this project, I continue working on **Lane 2: Refresh Opportunity Scoring**, which focuses on supporting editorial teams in identifying the content pages that should be reviewed first for potential updates.

From a machine learning perspective, I frame this problem as a **scoring task**. Rather than making a simple binary prediction such as *"refresh"* or *"do not refresh,"* the objective is to assign each content page a **Refresh Priority Score** that represents its relative urgency for editorial review.

I selected a scoring approach because the business problem is not deciding whether a page should be refreshed in isolation. Instead, editors must prioritize their limited time across approximately **30,000 content pages**. A priority score allows the pages to be ranked from highest to lowest urgency, enabling editorial teams to focus on the most valuable refresh opportunities first.

This framing is also consistent with the philosophy of a **decision support system**. The proposed model is intended to assist editors by providing an evidence-based priority ranking rather than replacing human judgment. Editors remain responsible for making the final refresh decision after considering the model's recommendations together with their domain expertise and business context.

----

# 2. Target or Proxy

----
The starter dataset does not contain a direct label indicating the editorial priority of each content page. In other words, there is no existing column that tells us which pages should be refreshed first. Therefore, instead of predicting a predefined target, I propose using a **Refresh Priority Score** as a **proxy target**.

The proposed Refresh Priority Score is intended to represent the relative urgency with which a content page should be reviewed by the editorial team. A higher score indicates that the page deserves earlier editorial attention, while a lower score suggests that the page can be reviewed later. This approach naturally supports the creation of a prioritized review queue rather than producing a simple binary decision.

I selected this proxy because the business objective is not to automatically decide whether a page should be refreshed. Instead, the objective is to help editors allocate their limited time more effectively across approximately 30,000 content pages. A priority score enables editors to review the highest-impact pages first while preserving human judgment in the final refresh decision.

Conceptually, the proposed proxy would be informed by multiple groups of observable content signals rather than a single feature. These signals include:

- **Business importance** (e.g., search volume, commercial value)
- **Search performance** (e.g., CTR, average position, trend direction)
- **User engagement** (e.g., sessions, engagement rate, scroll rate)
- **Content freshness** (e.g., content age, days since last update)

Rather than defining an exact mathematical formula at this stage, this notebook introduces the conceptual design of the proxy target. The relative contribution of these signals and the final scoring methodology will be investigated, validated, and refined during the later stages of the project.

Unlike a binary refresh recommendation, a priority score allows editorial teams to rank pages by urgency and determine how many pages they can realistically review within their available resources.

------

# 3. Success Metric

----

From a business perspective, the success of this project is not measured by whether the model simply produces a prediction. Instead, success is determined by whether the proposed Refresh Priority Score helps editorial teams identify the most valuable refresh opportunities earlier and more consistently than a purely manual review process.

A successful system should place the pages with the highest refresh potential near the top of the priority queue, allowing editors to focus their limited time on reviewing the pages that are most likely to benefit from updates. The objective is to improve editorial prioritization rather than automate editorial decision-making.

Another important measure of success is explainability. Editors should be able to understand why a page receives a high priority score by examining the observable content signals that influenced the recommendation. This transparency helps build trust in the system and supports informed human decision-making.

At this stage of the project, no machine learning model has been developed, so model-specific evaluation metrics are not yet appropriate. If a predictive model is implemented in later stages, suitable ranking-oriented evaluation metrics will be selected to assess how effectively the generated priority scores support editorial prioritization.

The objective is to improve the quality and consistency of editorial prioritization rather than maximizing a traditional prediction metric.

----




# 4. The Unit of Analysis

----

The unit of analysis for this project is **one content page**. Each row in the dataset represents a single content page together with its observable business, search performance, engagement, and freshness signals. These attributes describe the current condition of that page and provide the information required for estimating its refresh priority.

This page-level representation is appropriate because editorial refresh decisions are also made at the page level. Editors do not decide whether an entire client or website should be refreshed at once. Instead, they evaluate individual content pages and determine which ones deserve attention based on their current performance and business value.

Within the proposed machine learning framework, each content page is treated as one independent observation. For every page, the model receives its observable signals as input and generates a single **Refresh Priority Score**, indicating how urgently that page should be reviewed relative to the others.

To make this framing concrete, the following dataframe shows the unit of analysis used throughout the project. Each row corresponds to one content page, while each column represents one of the observable signals available for supporting editorial prioritization.



---

In [2]:
#import the pandas datafram
import pandas as pd

In [6]:
#laod the dataset into Pandas DataFrame
df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [5]:
#The displayed dataframe illustrates one content page together with the observable signals that will later contribute to estimating its Refresh Priority Score.
lane2_columns = [
    "content_id",
    "client_id",
    "search_volume",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct",
    "content_age_days",
    "sessions_90d",
    "engagement_rate"
]

df[lane2_columns].head()

,content_id,client_id,search_volume,ctr,avg_position,trend_direction,trend_pct,content_age_days,sessions_90d,engagement_rate
0,content_304f48230142,client_f369cb89fc,10.0,0.76,10.6,down,-41.4,187,17,5.88
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.05,20.3,down,-57.7,445,9,0.00
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.09,36.5,down,-60.9,141,11,0.00
3,content_331d6c4de07b,client_19581e27de,10.0,0.49,6.2,stable,-13.8,463,78,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.13,44.0,down,-34.7,263,145,0.00


**Note:** Although the dataset contains multiple pages belonging to the same client, each page is analyzed independently because refresh decisions are ultimately made at the content-page level.

# 5. Why ML Beats a Fixed Rule

---

A simple rule-based system could be created for this problem. For example, one might decide to refresh every page with a declining trend, low CTR, or old content. While such rules are easy to understand, they are often too rigid to support real editorial decision-making.

Content refresh priority depends on the interaction of multiple observable signals rather than a single condition. For example, a page with declining traffic may not deserve immediate attention if it has very low search demand, while another page with moderate performance decline but very high search volume could represent a much more valuable refresh opportunity. Fixed rules struggle to capture these complex relationships because they rely on manually defined thresholds and conditions.

A machine learning approach provides greater flexibility by learning patterns from multiple content signals simultaneously. Instead of evaluating each signal independently, the proposed system can consider business importance, search performance, user engagement, and content freshness together when estimating a page's refresh priority. This allows the generated recommendations to better reflect the complexity of real editorial decisions.

Finally, the objective of this project is not to replace editorial expertise with automated decisions. The Refresh Priority Score is intended to support editors by providing an evidence-based recommendation that helps prioritize review efforts. Editors remain responsible for making the final refresh decision using both the model's recommendation and their professional judgment.

Another advantage of a machine learning approach is that it can adapt as new data becomes available. Content performance, search trends, and user behavior change over time, making it difficult to maintain an effective set of fixed rules. A data-driven approach provides greater flexibility for capturing these changing patterns throughout the lifetime of the system.

The proposed machine learning system is expected to complement simple editorial heuristics instead of completely replace them.

---

# 6. Self Check

----

Before moving to the next stage of the project, I reviewed whether this notebook successfully translates the business problem into a well-defined machine learning problem.

- [x] I clearly identified the machine learning task as a **scoring problem** that supports editorial prioritization.

- [x] I proposed a **Refresh Priority Score** as a proxy target because the dataset does not contain a direct editorial priority label.

- [x] I defined success from both a business and machine learning perspective, emphasizing effective prioritization rather than simple prediction.

- [x] I identified **one content page** as the unit of analysis and explained why page-level prediction aligns with the editorial workflow.

- [x] I justified why a machine learning approach is more suitable than a small set of fixed rules for this problem.

- [x] I connected the model's output directly to a real editorial action by supporting, rather than replacing, human decision-making.

Overall, this notebook establishes a clear machine learning framing for the project before any feature engineering or model development begins. The next stage of the project will focus on exploring the data in greater depth and validating the proposed proxy target using evidence from the dataset.

----

## Reflection
---

Completing this notebook helped me understand that designing a machine learning system begins with carefully framing the prediction problem rather than selecting a model. By defining the task type, proxy target, success criteria, and unit of analysis before modeling, I now have a clear foundation for the remaining stages of the project.

---